In [ ]:
## pull the data with links
# --- Initialize data accumulation storage BEFORE the loop ---
all_table_data = []
headers = []  # We will capture headers on the first successful pass

# -----------------------------
# 1. Get the total number of options first safely
# -----------------------------
state_dropdown = wait.until(
    EC.presence_of_element_located((By.XPATH, "//select[@formcontrolname='state']"))
)
total_options = len(Select(state_dropdown).options)
print(total_options)
# -----------------------------
# 2. Loop by index
# -----------------------------
for index in range(1, total_options):  # Start at 1 to skip placeholder
    #if index == 1:
        #continue
    state_dropdown = wait.until(
        EC.visibility_of_element_located((By.XPATH, "//select[@formcontrolname='state']"))
    )
    state_select = Select(state_dropdown)
    state_value = state_select.options[index].get_attribute("value")
    
    state_select.select_by_index(index)
    print(f"\nProcessing state index {index} (Value: {state_value})...")

    # Click Search Button Safely
    search_button = wait.until(
        EC.element_to_be_clickable((By.XPATH, "//button[@type='submit' and contains(.,'Search')]"))
    )
    
    existing_tables = driver.find_elements(By.ID, "excel-table")
    old_table = existing_tables[0] if existing_tables else None

    driver.execute_script("arguments[0].click();", search_button)
    print(f"Search successfully forced click for: {state_value}")
    
    if old_table:
        try:
            wait.until(EC.staleness_of(old_table))
        except Exception:
            time.sleep(1) 

    # Handle Missing Table if No Results Exist
    try:
        wait.until(EC.visibility_of_element_located((By.ID, "excel-table")))
        print(f"Table successfully loaded for state: {state_value}")
    except TimeoutException:
        print(f"⚠️ No results found (Timeout) for state: {state_value}. Skipping...")
        time.sleep(1)
        continue

    # -----------------------------
    # LIVE ROW CLICKING & SCRAPING LOGIC 
    # -----------------------------
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')
    table = soup.find("table", {"id": "excel-table"})
    
    if table:
        tbody = table.find("tbody")
        if tbody:
            rows_bs = tbody.find_all("tr")
            
            tbody_text = tbody.get_text(strip=True).lower()
            if "no record" in tbody_text or "no data" in tbody_text or not rows_bs:
                print(f"ℹ️ Table contains explicit empty notice for state: {state_value}. Skipping...")
                time.sleep(1)
                continue

            if not headers:
                thead = table.find("thead")
                if thead:
                    for th in thead.find_all("th"):
                        headers.append(th.get_text(strip=True))

            total_rows = len(rows_bs)
            print(f"Found {total_rows} proposals to process for state {state_value}.")

            # Loop through rows sequentially
            for row_idx in range(1, total_rows + 1):
                try:
                    cols_elements = driver.find_elements(By.XPATH, f"//table[@id='excel-table']/tbody/tr[{row_idx}]/td")
                    row_data = [col.text.strip() for col in cols_elements]
                    
                    if not row_data:
                        continue
                    
                    proposal_link = driver.find_element(By.XPATH, f"//table[@id='excel-table']/tbody/tr[{row_idx}]/td[2]/a")
                    proposal_no = proposal_link.text.strip()
                    
                    print(f" -> Clicking proposal {row_idx}/{total_rows}: {proposal_no}")
                    
                    main_window = driver.current_window_handle
                    driver.execute_script("arguments[0].click();", proposal_link)
                    time.sleep(3)  
                    
                    details_url = "N/A"
                    view_proposal_url = "N/A"
                    
                    opened_in_new_tab = len(driver.window_handles) > 1
                    if opened_in_new_tab:
                        details_window = [w for w in driver.window_handles if w != main_window][0]
                        driver.switch_to.window(details_window)
                    
                    details_url = driver.current_url
                    
                    # -------------------------------------------------------------
                    # NESTED EXTRACTION: Click "View Proposal" & Capture Document URL
                    # -------------------------------------------------------------
                    try:
                        view_proposal_element = driver.find_element(By.XPATH, "//a[contains(translate(text(), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'view proposal')]")
                        pre_click_windows = driver.window_handles
                        
                        driver.execute_script("arguments[0].click();", view_proposal_element)
                        time.sleep(3)
                        
                        post_click_windows = driver.window_handles
                        if len(post_click_windows) > len(pre_click_windows):
                            proposal_doc_window = [w for w in post_click_windows if w not in pre_click_windows][0]
                            driver.switch_to.window(proposal_doc_window)
                            
                            view_proposal_url = driver.current_url  
                            
                            driver.close()  # Close document tab
                            if opened_in_new_tab:
                                driver.switch_to.window(details_window)
                            else:
                                driver.switch_to.window(main_window)
                        else:
                            view_proposal_url = driver.current_url
                            driver.back()
                            time.sleep(1)
                            
                    except NoSuchElementException:
                        print("    ⚠️ 'View Proposal' link/button was not found on this view layout.")
                    except Exception as nested_err:
                        print(f"    ❌ Failed tracking 'View Proposal' sub-route: {nested_err}")
                    
                    # -------------------------------------------------------------
                    # BACKWARDS ROUTING CLEANUP & RE-ALIGNMENT
                    # -------------------------------------------------------------
                    if opened_in_new_tab:
                        driver.close() 
                        driver.switch_to.window(main_window)
                    else:
                        driver.back() 
                        wait.until(EC.visibility_of_element_located((By.ID, "excel-table")))
                    
                    # Compile target URL metrics back into data array rows
                    row_data.append(details_url)        
                    row_data.append(view_proposal_url)   
                    row_data.append(state_value)        
                    all_table_data.append(row_data)
                    print(f"    Successfully Scraped URLs.")
                    
                except NoSuchElementException:
                    print(f" ⚠️ Could not find target link anchor for index row {row_idx}. Skipping...")
                    continue
                except Exception as e:
                    print(f" ❌ Error processing index row {row_idx}: {e}")
                    all_windows = driver.window_handles
                    if len(all_windows) > 1:
                        for extra_w in all_windows[1:]:
                            driver.switch_to.window(extra_w)
                            driver.close()
                    driver.switch_to.window(main_window)
                    continue

    time.sleep(1)
    #break  # Kept active for test validation. Remove or comment out to run for all states.

# -----------------------------
# 3. Post-Loop Data Compilation
# -----------------------------
if all_table_data:
    expected_header_count = len(all_table_data[0])
    
    if len(headers) < expected_header_count:
        headers.append("Details_URL")      
        headers.append("Proposal_URL")     
        headers.append("State_Value")
        
    df = pd.DataFrame(all_table_data, columns=headers)
    print("\n--- Final Extracted Dataset Preview ---")
    print(df.head()) 

    file_path = "parivesh_data.csv"

    # Check if the file already exists
    file_exists = os.path.exists(file_path)

    # Write to CSV
    df.to_csv(
    file_path, 
    mode='a', 
    index=False, 
    header=not file_exists  # Writes header ONLY if the file does NOT exist yet
)
    print("\nAll available states scraped and saved to parivesh_data.csv successfully!")
else:
    print("\n❌ Automation complete. Zero data entries found across all processed states.")

In [ ]:
#work in progress.

import os
import re
import pandas as pd
from dotenv import load_dotenv

load_dotenv()


# Define Taxonomy Dictionary (Main Headings without numbering)
TAXONOMY = {
    "Metallic Minerals": {
        "Iron Ore": [
            "Iron Ore",
            "Hematite",
            "Magnetite",
            "Siderite",
            "Goethite",
            "Iron Sand",
        ],
        "Bauxite": [
            "Bauxite",
            "Aluminium Ore",
            "Alumina Ore",
            "Hydrous Aluminum Oxide",
        ],
        "Copper Ore": [
            "Copper Ore",
            "Copper",
            "Chalcopyrite",
            "Bornite",
            "Malachite",
            "Cuprite",
        ],
        "Manganese Ore": [
            "Manganese Ore",
            "Manganese",
            "Pyrolusite",
            "Psilomelane",
            "Rhodochrosite",
        ],
        "Chromite": ["Chromite", "Chromium Ore", "Ferrochrome Ore"],
        "Gold": [
            "Gold",
            "Native Gold",
            "Gold Ore",
            "Auriferous Quartz",
            "Placer Gold",
        ],
        "Silver": ["Silver", "Native Silver", "Argentite", "Galvanic Silver"],
        "Lead & Zinc": [
            "Lead & Zinc",
            "Lead",
            "Zinc",
            "Galena",
            "Sphalerite",
            "Zinc Blende",
            "Calamine",
        ],
        "Nickel": ["Nickel", "Lateritic Nickel", "Pentandite"],
        "Titanium Ores": [
            "Titanium Ores",
            "Titanium",
            "Ilmenite",
            "Rutile",
            "Titaniferous Magnetite",
        ],
        "Tin": ["Tin", "Cassiterite", "Tinstone"],
    },
    "Energy & Strategic / Atomic Minerals": {
        "Coal": [
            "Coal",
            "Thermal Coal",
            "Coking Coal",
            "Bituminous",
            "Lignite",
            "Brown Coal",
            "Anthracite",
        ],
        "Petroleum & Natural Gas": [
            "Petroleum & Natural Gas",
            "Petroleum",
            "Crude Oil",
            "Mineral Oil",
            "Hydrocarbons",
            "Natural Gas",
            "Shale Gas",
        ],
        "Uranium": ["Uranium", "Pitchblende", "Uraninite"],
        "Thorium": ["Thorium", "Monazite Sand", "Thorianite"],
        "Lithium": ["Lithium", "Spodumene", "Lepidolite", "White Gold"],
    },
    "Non-Metallic & Industrial Minerals": {
        "Limestone": [
            "Limestone",
            "Calcium Carbonate",
            "Calcite",
            "Chalk",
            "Quicklime Stone",
        ],
        "Dolomite": ["Dolomite", "Dolostone", "Magnesium Limestone"],
        "Mica": [
            "Mica",
            "Muscovite",
            "Phlogopite",
            "Biotite",
            "Isinglass",
            "Sheet Mica",
            "Mica Flakes",
        ],
        "Gypsum": ["Gypsum", "Selenite", "Alabaster", "Hydrated Calcium Sulfate"],
        "Rock Phosphate": ["Rock Phosphate", "Phosphorite", "Apatite", "Phosphate Rock"],
        "Fluorspar": ["Fluorspar", "Fluorite", "Calcium Fluoride"],
        "Barite": ["Barite", "Barytes", "Heavy Spar", "Barium Sulfate"],
        "Magnesite": ["Magnesite", "Magnesium Carbonate"],
        "Kyanite & Sillimanite": [
            "Kyanite & Sillimanite",
            "Kyanite",
            "Sillimanite",
            "Aluminosilicate Minerals",
            "Refractory Minerals",
        ],
        "Asbestos": ["Asbestos", "Chrysotile", "Amphibole", "White Asbestos"],
        "Feldspar": ["Feldspar", "Orthoclase", "Plagioclase", "Microcline"],
        "Quartz & Silica": [
            "Quartz & Silica",
            "Silica Sand",
            "Quartzite",
            "Crystal Quartz",
            "Rock Crystal",
        ],
        "Talc & Soapstone": [
            "Talc & Soapstone",
            "Talc",
            "Soapstone",
            "Steatite",
            "French Chalk",
            "Hydrous Magnesium Silicate",
        ],
    },
    "Gemstones & Precious Stones": {
        "Diamond": [
            "Diamond",
            "Carbon Crystal",
            "Gem Diamond",
            "Industrial Diamond",
            "Bort",
        ],
        "Emerald": ["Emerald", "Green Beryl"],
        "Ruby & Sapphire": [
            "Ruby & Sapphire",
            "Ruby",
            "Sapphire",
            "Corundum",
            "Manik",
            "Neelam",
        ],
        "Garnet": ["Garnet", "Almandine", "Pyrope", "Garnet Sand"],
        "Agate & Chalcedony": [
            "Agate & Chalcedony",
            "Agate",
            "Chalcedony",
            "Carnelian",
            "Jasper",
            "Onyx",
            "Silica Stones",
        ],
    },
    "Building & Dimension Stones": {
        "Granite": [
            "Granite",
            "Dimension Stone",
            "Commercial Granite",
            "Black Granite",
        ],
        "Marble": [
            "Marble",
            "Makrana Marble",
            "Calcitic Marble",
            "Dolomitic Marble",
            "Crystalline Limestone",
        ],
        "Sandstone": [
            "Sandstone",
            "Red Fort Stone",
            "Dholpur Stone",
            "Buff Sandstone",
            "Quartzose Sandstone",
        ],
        "Slate & Schist": [
            "Slate & Schist",
            "Slate",
            "Schist",
            "Roofing Slate",
            "Flagstone",
        ],
        "Laterite": ["Laterite", "Laterite Stone", "Building Block Stone"],
        "Basalt & Trap Rock": [
            "Basalt & Trap Rock",
            "Basalt",
            "Trap Rock",
            "Deccan Trap",
            "Black Trap",
            "Crushed Stone",
        ],
    },
    "Sand, Gravel & Aggregates": {
        "River Sand": ["River Sand", "Natural Sand", "Coarse Sand", "Bajri", "Reti"],
        "Silica Sand": ["Silica Sand", "Glass Sand", "Industrial Sand", "Quartz Sand"],
        "Manufactured Sand": [
            "Manufactured Sand",
            "M-Sand",
            "Crushed Rock Sand",
            "Artificial Sand",
        ],
        "Gravel & Aggregate": [
            "Gravel & Aggregate",
            "Gravel",
            "Aggregate",
            "Jit",
            "Gitti",
            "Metal Stones",
            "Crushed Stone Aggregate",
        ],
        "Placer & Beach Sands": [
            "Placer & Beach Sands",
            "Monazite Sand",
            "Heavy Mineral Sand",
            "Black Sand",
        ],
    },
}


def classify_text(text, taxonomy):
    """
    Scans project text for mineral keywords and extracts Main Heading, Subheading, and Keyword.
    If multiple minerals are found, joins them using comma separation.
    """
    if not isinstance(text, str) or not text.strip():
        return "Unclassified", "Unclassified", "None"

    # Flatten taxonomy keywords and sort by keyword length descending (longer phrases match first)
    keyword_map = []
    for main_heading, subheadings in taxonomy.items():
        for subheading, keywords in subheadings.items():
            all_kw = set([subheading] + keywords)
            for kw in all_kw:
                keyword_map.append((kw, main_heading, subheading))

    keyword_map.sort(key=lambda x: len(x[0]), reverse=True)

    matched_main = []
    matched_sub = []
    matched_kw = []
    seen_subheadings = set()

    for kw, main_h, sub_h in keyword_map:
        # Regex search using word boundaries (\b) and case insensitivity
        pattern = r"\b" + re.escape(kw) + r"\b"
        if re.search(pattern, text, flags=re.IGNORECASE):
            if (main_h, sub_h) not in seen_subheadings:
                seen_subheadings.add((main_h, sub_h))
                matched_main.append(main_h)
                matched_sub.append(sub_h)
                matched_kw.append(kw)

    if matched_main:
        # Deduplicate main headings while preserving order
        unique_mains = list(dict.fromkeys(matched_main))
        return (
            ", ".join(unique_mains),
            ", ".join(matched_sub),
            ", ".join(matched_kw),
        )
    else:
        return "Unclassified", "Unclassified", "None"


def update_excel_file(file_path, project_col_name="Project Name"):
    """
    Reads the Excel file, adds/updates classification columns, and saves back to the same path.
    """
    if not os.path.exists(file_path):
        print(f"Error: File not found at path: {file_path}")
        return

    print(f"Loading Excel file: {file_path} ...")
    df = pd.read_excel(file_path)

    # Check if the project column exists (case-insensitive check)
    target_col = None
    for col in df.columns:
        if col.strip().lower() == project_col_name.lower():
            target_col = col
            break

    if not target_col:
        print(
            f"Error: Column '{project_col_name}' not found in Excel sheet. Available columns: {list(df.columns)}"
        )
        return

    print(f"Classifying project names using column '{target_col}' ...")

    # Apply classification row-by-row
    classifications = df[target_col].apply(
        lambda x: classify_text(x, TAXONOMY)
    )

    # Unpack classification results into separate lists
    main_headings = [c[0] for c in classifications]
    subheadings = [c[1] for c in classifications]
    keywords = [c[2] for c in classifications]

    # Add or update the new columns in the DataFrame
    df["Main Heading"] = main_headings
    df["Subheading"] = subheadings
    df["Matched Keyword"] = keywords

    # Save back to the same Excel file
    print(f"Saving updated DataFrame back to: {file_path} ...")
    df.to_excel(file_path, index=False)
    print("Process completed successfully!")


if __name__ == "__main__":
    # Your file path

   

# 1. Path Configuration
    excel_path = os.getenv("BRONZE") + r"\RAW_MERGED.xlsx"
    

    # Pass the path and your project column name if it differs from 'Project Name'
    update_excel_file(excel_path, project_col_name="Project Name")